In [1]:
# command to reload libraries
%load_ext autoreload
%autoreload 2

In [14]:
import desReg
import os
from pathlib import Path
from sklearn.model_selection import cross_validate
from desReg.utils.scoring import get_scorers
from desReg.des.DESRegression import DESRegression
from sklearn import linear_model
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import make_regression
import pandas as pd

from desReg.des.DESRegression import DESRegression
from desReg.utils import measures as em
from sklearn import linear_model
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold

In [3]:
REPO_ROOT = Path.cwd().resolve().parent
DATA_PATH = REPO_ROOT / "data"

In [4]:
def load_dataset(file_name):
    df = pd.read_csv(DATA_PATH / file_name)
    return df

def have_more_than_300_samples(dataset):
    return dataset.shape[0] > 300
    


In [5]:
all_datasets_dict = {
    file.strip(".csv"): load_dataset(file) \
    for file in os.listdir(DATA_PATH) if (DATA_PATH / file).is_file()
}

datasets_names = list(all_datasets_dict.keys())

In [6]:
datasets_names_greater_than_300 = list()
for name,dataset in all_datasets_dict.items():
    if have_more_than_300_samples(dataset):
        datasets_names_greater_than_300.append(name)
len(datasets_names_greater_than_300)


57

In [7]:
# Define a simple pool of base regressors
regressors_list = [
    linear_model.Lasso(),
    linear_model.Ridge(),
    KNeighborsRegressor(),
    SVR(),
    DecisionTreeRegressor()
]

# Initialize DES model (uses _predict_DES by default)
model = DESRegression(
    regressors_list=regressors_list,
    random_state=0,
    n_estimators_bag=20,
    competence_region='knn',
    k=5,
    include_instance_hardness=False  # set True only if your data is numeric and you want IH weighting
)

# Build multi-metric scorers (includes ensemble-size summaries)
scorers = get_scorers(include_ensemble_metrics=True)

In [15]:
X = all_datasets_dict[datasets_names_greater_than_300[0]].drop(columns=['target'])
y = all_datasets_dict[datasets_names_greater_than_300[0]]['target']

k_folds = KFold(n_splits=5, random_state=42, shuffle=True)
# Run cross-validation
cv_results = cross_validate(
    model, X, y,
    cv=k_folds,
    scoring=scorers,
    return_train_score=False,
    n_jobs=None,
)

# Summarize test metrics
metrics_df = pd.DataFrame({k: v for k, v in cv_results.items() if k.startswith('test_')})
metrics_summary = metrics_df.mean().to_frame('mean').T
metrics_summary


,test_neg_mse,test_neg_rmse,test_neg_mae,test_neg_mape,test_r2,test_mean_total_regressors,test_mean_distinct_regressors
mean,-101.655912,-9.91395,-6.532639,-6.371629e+15,0.096721,53.435665,4.402282


In [32]:
X = all_datasets_dict['ocomo_numeri'].drop(columns=['target'])
y = all_datasets_dict['ocomo_numeri']['target']

# Run cross-validation
cv_results = cross_validate(
    model, X, y,
    cv=k_folds,
    scoring=scorers,
    return_train_score=False,
    n_jobs=None,
)

# Summarize test metrics
metrics_df = pd.DataFrame({k: v for k, v in cv_results.items() if k.startswith('test_')})
metrics_summary = metrics_df.mean().to_frame('mean').T
metrics_summary


desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegression - INFO - Predicting DES
desReg.des.DESRegres

,test_neg_mse,test_neg_rmse,test_neg_mae,test_neg_mape,test_r2,test_mean_total_regressors,test_mean_distinct_regressors
mean,-101899.366132,-264.434439,-148.98181,-0.389857,0.734905,68.883333,4.75


In [34]:
cv_results

{'fit_time': array([5.71788311, 0.07445407, 0.08976507, 0.07749295, 0.07371402]),
 'score_time': array([2.1830678 , 2.09406495, 2.23776889, 1.98502898, 1.91879797]),
 'test_neg_mse': array([-246417.84315369,   -1621.41746718, -119970.75178118,
        -136663.13685068,   -4823.68140923]),
 'test_neg_rmse': array([-496.40491854,  -40.26682837, -346.36794277, -369.67977609,
         -69.45272787]),
 'test_neg_mae': array([-301.44663178,  -32.75159482, -148.42027817, -217.094503  ,
         -45.19604122]),
 'test_neg_mape': array([-0.42628789, -0.37313957, -0.29367251, -0.5316086 , -0.32457716]),
 'test_r2': array([0.73881086, 0.92136942, 0.28257755, 0.80006689, 0.93169965]),
 'test_mean_total_regressors': array([69.        , 69.08333333, 67.25      , 68.5       , 70.58333333]),
 'test_mean_distinct_regressors': array([4.16666667, 5.        , 4.83333333, 4.75      , 5.        ])}

In [16]:
model.predict(X.loc[0:2], return_ensemble_sizes=True)

([9.773316354265292, 7.115260071489987, 9.183367987076556],
 [Counter({'TotalRegressors': 66,
           'Lasso': 20,
           'KNeighborsRegressor': 15,
           'SVR': 15,
           'Ridge': 8,
           'DecisionTreeRegressor': 8,
           'DistinctRegressors': 5}),
  Counter({'TotalRegressors': 55,
           'SVR': 20,
           'DecisionTreeRegressor': 19,
           'KNeighborsRegressor': 16,
           'DistinctRegressors': 3}),
  Counter({'TotalRegressors': 78,
           'Lasso': 20,
           'Ridge': 20,
           'SVR': 20,
           'DecisionTreeRegressor': 12,
           'KNeighborsRegressor': 6,
           'DistinctRegressors': 5})])

In [18]:
from sklearn.model_selection import PredefinedSplit

def get_predefined_folds(dataset_name, partition_base_path, n_folds=5):
    """
    Load predefined train/test splits from .dat files and create a PredefinedSplit CV object.
    
    Parameters
    ----------
    dataset_name : str
        Name of the dataset (e.g., 'abalone')
    partition_base_path : str
        Base path pattern for the dataset files (e.g., './Datasets/Abalone/abalone-5-')
    n_folds : int, default=5
        Number of folds to load
    
    Returns
    -------
    X : pd.DataFrame or np.ndarray
        Combined feature data from all folds
    y : pd.Series or np.ndarray
        Combined target data from all folds
    cv_splitter : PredefinedSplit
        Cross-validator with predefined fold assignments
    
    Example
    -------
    >>> X, y, cv = get_predefined_folds('abalone', './Datasets/Abalone/abalone-5-', n_folds=5)
    >>> cv_results = cross_validate(model, X, y, cv=cv, scoring=scorers)
    """
    all_X = []
    all_y = []
    fold_indices = []
    
    for fold in range(1, n_folds + 1):
        # Load training data
        train_file = f"{partition_base_path}{fold}tra.dat"
        data_train = pd.read_csv(train_file, header=None)
        X_train = data_train.iloc[:, :-1]
        y_train = data_train.iloc[:, -1]
        
        # Load test data
        test_file = f"{partition_base_path}{fold}tst.dat"
        data_test = pd.read_csv(test_file, header=None)
        X_test = data_test.iloc[:, :-1]
        y_test = data_test.iloc[:, -1]
        
        # Append to combined dataset
        all_X.append(X_train)
        all_X.append(X_test)
        all_y.append(y_train)
        all_y.append(y_test)
        
        # Assign fold indices: -1 for train (excluded from validation), fold number for test
        fold_indices.extend([-1] * len(X_train))  # Train samples (not used for validation in this fold)
        fold_indices.extend([fold - 1] * len(X_test))  # Test samples (used for validation in this fold)
    
    # Combine all data
    X = pd.concat(all_X, ignore_index=True)
    y = pd.concat(all_y, ignore_index=True)
    
    # Create PredefinedSplit object
    cv_splitter = PredefinedSplit(test_fold=fold_indices)
    
    return X, y, cv_splitter

In [24]:

# Usage example:
partitions_dict = {
    'abalone': '../Examples/Datasets/Abalone/abalone-5-',
    'yacht': '../Examples/Datasets/Yacht/yacht_hydrodynamics-5-',
    # ... other datasets
}

datasets_with_predefined_folds = ['abalone', 'yacht', 'liver', 'machineCPU']

# In your notebook or script:
dataset_name = 'yacht'
des_baseline_ensemble = DESRegression(
    regressors_list=regressors_list,
    random_state=0,
    n_estimators_bag=20,
    competence_region='knn',
    k=5,
    competence_level=em.all_errors,
    include_instance_hardness=False
)

if dataset_name in datasets_with_predefined_folds:
    X, y, predefined_cv = get_predefined_folds(
        dataset_name, 
        partitions_dict[dataset_name], 
        n_folds=5
    )
else:
    # Use regular k-fold
    cv = 5  # or KFold(n_splits=5, shuffle=True, random_state=0)

# Run cross-validation with either predefined or regular folds
cv_results = cross_validate(
    des_baseline_ensemble, X, y,
    cv=predefined_cv,
    scoring=scorers,
    return_train_score=False
)

In [31]:
predefined_cv

PredefinedSplit(test_fold=array([-1, -1, ...,  4,  4]))

In [30]:
cv_results

{'fit_time': array([0.242028  , 0.12687206, 0.12731123, 0.12368393, 0.20436192]),
 'score_time': array([10.33090901, 10.28610206, 10.57737708, 10.31403995, 10.14012218]),
 'test_neg_mse': array([ -8.85364801, -10.39655918,  -9.29238597, -10.62227157,
         -9.67078131]),
 'test_neg_rmse': array([-2.97550803, -3.22436958, -3.04834151, -3.25918265, -3.10978798]),
 'test_neg_mae': array([-1.34029066, -1.29825257, -1.52120869, -1.48266872, -1.53246364]),
 'test_neg_mape': array([-1.07239617, -0.53912443, -1.27151891, -1.55486672, -2.25126926]),
 'test_r2': array([0.95538839, 0.94369392, 0.9632343 , 0.95821035, 0.96162952]),
 'test_mean_total_regressors': array([64.25806452, 65.09677419, 63.80645161, 63.59016393, 63.75409836]),
 'test_mean_distinct_regressors': array([3.38709677, 3.4516129 , 3.37096774, 3.40983607, 3.36065574])}

## Partitions

In [75]:
from desReg.des.DESRegression import DESRegression
from desReg.utils import measures as em
from sklearn import linear_model
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

regressors_list = [
    linear_model.Lasso(), 
    linear_model.Ridge(), 
    KNeighborsRegressor(), 
    SVR(), 
    DecisionTreeRegressor()
]

des_ensemble_ih = DESRegression(
    regressors_list = regressors_list,
    random_state=0,
    n_estimators_bag=20,
    competence_region='knn',
    k=5,
    competence_level=em.sum_absolute_error_weighted,
    include_instance_hardness=True,
    include_hardness_measures=False,
    hardness_measures_list=None,
    meta_feature='instance_hardness',
    ensemble_type='DES'
)

In [68]:
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, random_state=42, shuffle=True)

In [72]:
datasets_folds = dict()
for name, dataset in all_datasets_dict.items():
    X = dataset.drop(columns=['target'])
    y = dataset['target']
    datasets_folds[name] = dict()
    for fold_number, (train_index, test_index) in enumerate(kf.split(X)):
        datasets_folds[name][fold_number+1] = dict(
            X_train = X.iloc[train_index],
            y_train = y.iloc[train_index],
            X_test = X.iloc[test_index],
            y_test = y.iloc[test_index]
        )


In [80]:
from sklearn.model_selection import cross_val_score

regressors_list = [
    linear_model.Lasso(), 
    linear_model.Ridge(), 
    KNeighborsRegressor(), 
    SVR(), 
    DecisionTreeRegressor()
]

des_ensemble_ih = DESRegression(
    regressors_list = regressors_list,
    random_state=0,
    n_estimators_bag=20,
    competence_region='knn',
    k=5,
    competence_level=em.sum_absolute_error_weighted,
    include_instance_hardness=True,
    include_hardness_measures=False,
    hardness_measures_list=None,
    meta_feature='instance_hardness',
    ensemble_type='DES'
)

X = all_datasets_dict[datasets_names_greater_than_300[0]].drop(columns=['target'])
y = all_datasets_dict[datasets_names_greater_than_300[0]]['target']
scores = cross_val_score(des_ensemble_ih, X, y, cv=5)
scores

desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Calculating instance hardness
desReg.des.DESRegression - INFO - Calculating instance hardness


array([ 78.56706599, 140.18603286, 142.76998691,  62.89311354,
        96.40584762])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 316 entries, 0 to 315
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   onehotencoder__x0_autumn  316 non-null    float64
 1   onehotencoder__x0_spring  316 non-null    float64
 2   onehotencoder__x0_summer  316 non-null    float64
 3   onehotencoder__x0_winter  316 non-null    float64
 4   onehotencoder__x1_large_  316 non-null    float64
 5   onehotencoder__x1_medium  316 non-null    float64
 6   onehotencoder__x1_small_  316 non-null    float64
 7   onehotencoder__x2_high__  316 non-null    float64
 8   onehotencoder__x2_low___  316 non-null    float64
 9   onehotencoder__x2_medium  316 non-null    float64
 10  concentration_1           316 non-null    float64
 11  concentration_2           316 non-null    float64
 12  concentration_3           316 non-null    float64
 13  concentration_4           316 non-null    float64
 14  concentrat